In [1]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=r"C:\Users\Lenovo\Downloads\4classes.yolov8 (1)\data.yaml",
    epochs=50,        # reduce first
    patience=20,
    imgsz=640,        # safer
    batch=8,          # safer
    mosaic=1.0,
    device=0,        
    project='models/traffic_v2',
    name='train'
)

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.36  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Lenovo\Downloads\4classes.yolov8 (1)\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001A302E76EF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

In [1]:
import os
import cv2
import json
import csv
import datetime
import numpy as np
from pathlib import Path
from collections import Counter

from ultralytics import YOLO

from src.detection import YOLODetector, Detection
from src.tracking import build_tracker, TrajectoryStore
from src.violations import ViolationEngine
from src.ocr import PlateRecognizer

BASE_DIR = Path(".")

OUTPUT_DIR = BASE_DIR / "outputs"
VIDEO_DIR  = OUTPUT_DIR / "videos"
REPORT_DIR = OUTPUT_DIR / "reports"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

print("✅ Setup ready")

✅ Setup ready


In [2]:
MODEL_PATH = "C:/Users/Lenovo/runs/detect/models/traffic_v2/train7/weights/best.pt"
VIDEO_PATH = "D:/e-challan-system/data/input/video3.mp4"

# Your trained model
detector = YOLODetector(
    weights=MODEL_PATH,
    conf=0.3,
    device="0"
)

# COCO model (for traffic light + vehicles)
coco_model = YOLO("yolov8n.pt")

print("✅ Models loaded")

[20:19:31] INFO [tvd.detection] Loading YOLO model: C:/Users/Lenovo/runs/detect/models/traffic_v2/train7/weights/best.pt on device=0
[20:19:32] INFO [tvd.detection] Model loaded successfully.


✅ Models loaded


In [3]:
video_out_path = VIDEO_DIR / f"annotated_{ts}.mp4"
json_out_path  = REPORT_DIR / f"violations_{ts}.json"
csv_out_path   = REPORT_DIR / f"violations_{ts}.csv"

def build_summary(records):
    return dict(Counter([r["violation"] for r in records]))

In [4]:
tracker = build_tracker("deepsort", max_age=50, n_init=1)
traj = TrajectoryStore()

plate_reader = PlateRecognizer(use_gpu=True)

track_memory = {}
IOU_THRESHOLD = 0.5

def iou(boxA, boxB):
    x1 = max(boxA[0], boxB[0])
    y1 = max(boxA[1], boxB[1])
    x2 = min(boxA[2], boxB[2])
    y2 = min(boxA[3], boxB[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0

    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    return inter / (areaA + areaB - inter)

print("✅ Tracker ready")

[20:19:32] INFO [tvd.tracking] DeepSORT tracker initialised.
[20:19:32] INFO [tvd.ocr] Initializing EasyOCR for Plate Recognition...


✅ Tracker ready


In [5]:
def auto_find_stop_line(frame):
    h, w = frame.shape[:2]

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    mask = np.zeros_like(edges)
    pts = np.array([[(0, int(h*0.4)), (w, int(h*0.4)), (w, h), (0, h)]], dtype=np.int32)
    cv2.fillPoly(mask, pts, 255)
    masked = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(masked, 1, np.pi/180, 100, minLineLength=w//4, maxLineGap=50)

    if lines is None:
        return (0, int(h*0.7), w, int(h*0.7))

    best = max(lines, key=lambda l: np.linalg.norm(l[0][:2]-l[0][2:]))
    x1,y1,x2,y2 = best[0]

    if x1 > x2:
        x1,y1,x2,y2 = x2,y2,x1,y1

    print("🎯 Stop line:", (x1,y1,x2,y2))
    return (x1,y1,x2,y2)

In [6]:
results = {
    "records": [],
    "summary": {},
    "video_out": str(video_out_path)
}

seen = set()

if os.path.exists(VIDEO_PATH):

    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Detect stop line once
    ret, first_frame = cap.read()
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    STOP_LINE = auto_find_stop_line(first_frame)

    # Init engine
    engine = ViolationEngine({
        "fps": fps,
        "stop_line": STOP_LINE
    })

    out = cv2.VideoWriter(
        str(video_out_path),
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

    frame_no = 0
    print("🚀 Processing video...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # ---------- DETECTION ----------
        custom_dets = detector.detect(frame)
        custom_dets = [d for d in custom_dets if d.conf >= 0.5]

        coco_results = coco_model(frame, conf=0.3)

        coco_dets = []
        for r in coco_results:
            for box in r.boxes:
                cls = int(box.cls[0])
                label = coco_model.names[cls]

                if label in ["traffic light", "car", "bus", "truck", "motorcycle", "bicycle"]:
                    x1,y1,x2,y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])

                    group = "traffic_light" if label == "traffic light" else "vehicle"

                    coco_dets.append(
                        Detection(
                            box=[x1,y1,x2,y2],
                            conf=conf,
                            class_id=cls,
                            class_name=label,
                            group=group
                        )
                    )

        detections = custom_dets + coco_dets

        # ---------- TRACKING ----------
        tracked = tracker.update(detections, frame)

        for det in tracked:
            if det.track_id != -1:
                traj.update(det.track_id, frame_no, det.center)

        # ---------- STATIC VIOLATIONS ----------
        for det in tracked:
            tid = det.track_id
            label = det.class_name
            box = det.box
            x1,y1,x2,y2 = map(int, box)

            key = (tid, label)

            if label in ["WITHOUT_HELMET", "USING_MOBILE", "MORE_THAN_TWO_PERSONS"]:
                if key not in seen:
                    seen.add(key)
                    
                    plate_text = plate_reader.process_vehicle(tid, frame, box)
                    
                    results["records"].append({
                        "vehicle_id": tid,
                        "frame": frame_no,
                        "violation": label,
                        "plate_number": plate_text
                    })

            color = (0,255,0)
            if label == "WITHOUT_HELMET": color = (0,0,255)
            elif label == "USING_MOBILE": color = (255,0,0)
            elif label == "MORE_THAN_TWO_PERSONS": color = (0,255,255)

            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,f"{tid}-{label}",(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

        # Cleanup old tracks to prevent memory leaks
        traj.cleanup_old_tracks(frame_no)
        
        # ---------- RED LIGHT ----------
        engine_records = engine.process_frame(frame, frame_no, tracked, traj)

        for r in engine_records:
            vid = r.get("vehicle_id")
            vbox = r.get("box")
            
            if vid is not None and vbox is not None:
                r["plate_number"] = plate_reader.process_vehicle(vid, frame, vbox)
            else:
                r["plate_number"] = None
                
            results["records"].append(r)

            if "box" in r:
                x1,y1,x2,y2 = map(int, r["box"])
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,0,255),4)
                cv2.putText(frame,"RED LIGHT!",(x1,y1-30),
                            cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),3)

        # Draw stop line
        cv2.line(frame,
                 (STOP_LINE[0],STOP_LINE[1]),
                 (STOP_LINE[2],STOP_LINE[3]),
                 (0,165,255),3)

        out.write(frame)
        cv2.imshow("Detection", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_no += 1

    cap.release()
    out.release()
    cv2.destroyAllWindows()

else:
    print("⚠️ Video not found")

[20:19:34] INFO [tvd.violations] ViolationEngine initialised with 1 detectors.


🎯 Stop line: (287, 934, 1797, 473)
🚀 Processing video...

0: 384x640 18 persons, 1 car, 8 motorcycles, 52.9ms
Speed: 1.7ms preprocess, 52.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 1 car, 8 motorcycles, 21.1ms
Speed: 3.5ms preprocess, 21.1ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 1 car, 8 motorcycles, 20.3ms
Speed: 2.7ms preprocess, 20.3ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 1 car, 8 motorcycles, 16.8ms
Speed: 2.3ms preprocess, 16.8ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 2 cars, 8 motorcycles, 16.2ms
Speed: 2.8ms preprocess, 16.2ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 2 cars, 9 motorcycles, 15.1ms
Speed: 3.0ms preprocess, 15.1ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 2 cars, 8 motorcy

In [7]:
results["summary"] = build_summary(results["records"])

with open(json_out_path,"w") as f:
    json.dump(results,f,indent=4)

with open(csv_out_path,"w",newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["vehicle_id","frame","violation"])
    for r in results["records"]:
        writer.writerow([r.get("vehicle_id"),r.get("frame"),r.get("violation")])

print("✅ DONE")
print("Summary:", results["summary"])

✅ DONE
Summary: {'MORE_THAN_TWO_PERSONS': 3}
